    # Goals
    • Which input features are likely useful? 
        ○ past load values 
        ○ hour of day 
        ○ day of week 
        ○ weekend 
        ○ holiday indicator  -> external dataset with public holidays necessary
        ○ Month
        ○ season (Winter = Dec, Jan, Feb / Spring = Mar, Apr, May / Summer = Jun, Jul, Aug / Autumn = Sep, Oct, Nov)
        ○ weather variables (if available) -> maybe in an extension
        ○ renewable generation (solar / wind)
    • Do we need feature engineering such as lag features, rolling averages, or cyclical encodings? 
        ○ Lag features (t-1 = load 1 hour ago / t-2 = load 2 hours ago / t-24 = load 24 hours ago / t-168 = load 168 hours ago (= 1 week ago)
        ○ Rolling statistics (rolling mean 24h)
        - Cyclical encoding for time-based variables (hour of the day, day of week, month)

In [37]:
#!pip install holidays
import pandas as pd
import numpy as np
import holidays
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)

In [38]:
df = pd.read_excel("../../data/austria/raw/austria_clean_master.xlsx")
df.head()

,timestamp,load_actual,load_forecast,price,solar,wind
0,2015-01-01 00:00:00+01:00,NaN,NaN,NaN,NaN,NaN
1,2015-01-01 01:00:00+01:00,NaN,NaN,NaN,NaN,NaN
2,2015-01-01 02:00:00+01:00,5750.8,6632.80,64.00,NaN,62.23
3,2015-01-01 03:00:00+01:00,5407.2,6507.80,46.82,NaN,66.70
4,2015-01-01 04:00:00+01:00,5284.8,6445.22,22.60,NaN,62.83


In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50401 entries, 0 to 50400
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   timestamp      50401 non-null  object 
 1   load_actual    50399 non-null  float64
 2   load_forecast  50399 non-null  float64
 3   price          50399 non-null  float64
 4   solar          50392 non-null  float64
 5   wind           50399 non-null  float64
dtypes: float64(5), object(1)
memory usage: 2.3+ MB


In [40]:
# 2. Convert timestamp from string (object) to datetime objects
# utc=True handles the timezone offsets correctly during conversion
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)

# 3. Convert to local Austrian time
# This is crucial for correctly identifying local hours, weekends, and holidays
df['timestamp'] = df['timestamp'].dt.tz_convert('Europe/Vienna')

# 4. Set timestamp as the index
# A datetime index is required for shifting (lags) and resampling
df = df.set_index('timestamp').sort_index()

In [41]:
df.head()

,load_actual,load_forecast,price,solar,wind
timestamp,,,,,
2015-01-01 00:00:00+01:00,NaN,NaN,NaN,NaN,NaN
2015-01-01 01:00:00+01:00,NaN,NaN,NaN,NaN,NaN
2015-01-01 02:00:00+01:00,5750.8,6632.80,64.00,NaN,62.23
2015-01-01 03:00:00+01:00,5407.2,6507.80,46.82,NaN,66.70
2015-01-01 04:00:00+01:00,5284.8,6445.22,22.60,NaN,62.83


In [42]:
# Define a function to create features for supervised learning

# 1. Function to map months to seasons
def get_season(month):
    """Maps month integers to season names."""
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'

# 2. Main Feature Engineering Pipeline
def build_modeling_features(df):
    """
    Transforms clean data into a supervised learning dataset.
    Adds time features, cyclical encoding, holidays, lags, and the target.
    """
    
    # Create a copy to avoid SettingWithCopy warnings
    df = df.copy()
    
    # --- Time-Based Features ---
    df['hour'] = df.index.hour
    df['day_of_week'] = df.index.dayofweek
    df['month'] = df.index.month
    
    # Cyclical Encoding (Captures the circular nature of time)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    
    df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

    # Night / Day Indicator
    # Definition: Nacht = 22:00–05:59
    df['is_night'] = df['hour'].apply(lambda x: 1 if (x >= 22 or x < 6) else 0)
    
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    
    # Weekend and Season
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['season'] = df['month'].apply(get_season)
    
    # Convert season to dummy variables (One-Hot Encoding)
    df = pd.get_dummies(df, columns=['season'], prefix='season')
    
    # --- Holiday Indicator (Austria) ---
    at_holidays = holidays.Austria()
    df['is_holiday'] = df.index.map(lambda x: 1 if x in at_holidays else 0).astype(int)
    
    # --- Lag Features (Past values as input) ---
    df['load_lag_1h'] = df['load_actual'].shift(1)
    df['load_lag_2h'] = df['load_actual'].shift(2)
    df['load_lag_24h'] = df['load_actual'].shift(24)
    df['load_lag_168h'] = df['load_actual'].shift(168)
    
    # --- Rolling Statistics ---
    # Mean load of the last 24 hours (shifted by 1 to avoid data leakage)
    df['load_rolling_mean_24h'] = df['load_actual'].shift(1).rolling(window=24).mean()
    
    # --- Target Definition ---
    # We want to forecast the load 24 hours ahead
    forecast_horizon = 24
    df['target_load_24h'] = df['load_actual'].shift(-forecast_horizon)
    
    # --- Data Cleaning ---
    # Convert any remaining booleans (from get_dummies) to integers (0/1)
    for col in df.columns:
        if df[col].dtype == 'bool':
            df[col] = df[col].astype(int)
            
    # Drop rows with NaNs (caused by lags at the start and the target shift at the end)
    df = df.dropna()
    
    return df


def time_split(df, train_ratio=0.7, val_ratio=0.15):
    n = len(df)

    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    train = df.iloc[:train_end]
    val = df.iloc[train_end:val_end]
    test = df.iloc[val_end:]

    return train, val, test

In [43]:
# Apply the pipeline
df_model = build_modeling_features(df)

print("Supervised learning dataset created!")
print(f"Number of features: {len(df_model.columns)}")

Supervised learning dataset created!
Number of features: 27


In [44]:
print(df_model.head(15))

                           load_actual  load_forecast  price  solar    wind  \
timestamp                                                                     
2015-01-08 02:00:00+01:00       6330.0        6790.67  23.57   0.00  161.53   
2015-01-08 03:00:00+01:00       6364.0        6679.49  16.12   0.00  165.50   
2015-01-08 04:00:00+01:00       6423.6        6710.88   3.51   0.00  169.50   
2015-01-08 05:00:00+01:00       6539.6        7025.10   1.05   0.00  183.31   
2015-01-08 06:00:00+01:00       7105.2        7505.86  -2.00   0.00  197.88   
2015-01-08 07:00:00+01:00       7819.6        8504.14  12.00   0.00  153.97   
2015-01-08 08:00:00+01:00       8096.8        8841.79  27.15   1.26  144.58   
2015-01-08 09:00:00+01:00       8334.4        8839.62  33.00  10.07  172.58   
2015-01-08 10:00:00+01:00       7555.6        8896.85  32.21  28.76  248.96   
2015-01-08 11:00:00+01:00       7600.0        8926.68  32.00  46.27  295.44   
2015-01-08 12:00:00+01:00       7834.4        8805.0

In [45]:
train_df, val_df, test_df = time_split(df_model)
#defining features and target for modeling
target = "target_load_24h"

features = [col for col in train_df.columns if col != target]

def scale_data(train, val, test, feature_cols):
    scaler = StandardScaler()

    train[feature_cols] = scaler.fit_transform(train[feature_cols])
    val[feature_cols] = scaler.transform(val[feature_cols])
    test[feature_cols] = scaler.transform(test[feature_cols])

    return train, val, test, scaler

In [46]:
#windowing function to create sequences for LSTM
def create_sequences(df, feature_cols, target_col, input_window=168, horizon=24):
    X, y = [], []

    data = df[feature_cols].values
    target = df[target_col].values

    for i in range(len(df) - input_window - horizon):
        X.append(data[i:i+input_window])
        y.append(target[i+input_window:i+input_window+horizon])

    return np.array(X), np.array(y)

In [47]:
train_df, val_df, test_df, scaler = scale_data(
    train_df, val_df, test_df, features
)

X_train, y_train = create_sequences(train_df, features, target)
X_val, y_val = create_sequences(val_df, features, target)
X_test, y_test = create_sequences(test_df, features, target)

C:\Users\betti\AppData\Local\Temp\ipykernel_29908\2054142467.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train[feature_cols] = scaler.fit_transform(train[feature_cols])
C:\Users\betti\AppData\Local\Temp\ipykernel_29908\2054142467.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val[feature_cols] = scaler.transform(val[feature_cols])
C:\Users\betti\AppData\Local\Temp\ipykernel_29908\2054142467.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try 

In [48]:
# --- EXPORT FINAL DATASET ---

# Define the output path
# Using a relative path to the 'processed' data folder
output_path = "../../data/austria/raw/austria_modeling_table_24h_horizon.csv"

# Export to CSV
# We keep the index because it's our timestamp!
train_df.to_csv("../../data/austria/raw/train.csv", index=True)
val_df.to_csv("../../data/austria/raw/val.csv", index=True)
test_df.to_csv("../../data/austria/raw/test.csv", index=True)